In [ ]:
from langchain_groq import ChatGroq

from dotenv import load_dotenv
import os
load_dotenv()

if os.environ['GROQ_API_KEY']:
    print("API Key is set.")
else:
    raise ValueError("API Key is not set.")

OpenAI API Key is set.


In [7]:
llm = ChatGroq(model="llama-3.1-8b-instant")

In [11]:
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage

In [ ]:

class AgentState(TypedDict):
    messages: Annotated[list, add_messages]
    is_done: bool

In [10]:
def agent_node(state: AgentState) -> dict:
    response = llm.invoke(state["messages"])
    return {"messages": [response]}

def done_node(state: AgentState) -> dict:
    return {"is_done": True}

In [13]:

graph = StateGraph(AgentState)


graph.add_node("agent_node", agent_node)
graph.add_node("done_node", done_node)

graph.add_edge(START,"agent_node")

def route(state: AgentState) -> str:
    if state["is_done"]:
        return "end"
    return "done_node"

graph.add_conditional_edges(
    "agent_node",
    route,
    {
        "done_node": "done_node",
        "end": END
    }
)

graph.add_edge("done_node", END)


compiled_graph = graph.compile()

In [19]:
response=compiled_graph.invoke({
    "messages": [HumanMessage(content="What is 2 + 2?")],
    "is_done": False
})
response['messages'][-1].content

'The answer to 2 + 2 is 4.'